# Week 4 — Day 4: Code Generator + Open-Source Models + Gradio UI

Extends day 3 by adding open-source code-generation models (running locally via Ollama, or hosted via Groq and OpenRouter) and wraps the whole thing in a Gradio dropdown UI.

Models exercised:
- **Frontier:** GPT-5, Claude 4.5 Sonnet, Grok 4, Gemini 2.5 Pro
- **Open-source local (Ollama):** qwen2.5-coder, deepseek-coder-v2, gpt-oss:20b
- **Open-source hosted:** OpenAI gpt-oss-120b (Groq), Qwen3-Coder-30B (OpenRouter)

Compiling and running C++ locally is optional — use https://www.programiz.com/cpp-programming/online-compiler/ as an alternative.

## Setup

In [ ]:
import os
import io
import sys
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

for name, key in [('OpenAI', openai_api_key), ('Anthropic', anthropic_api_key),
                  ('Google', google_api_key), ('Grok', grok_api_key),
                  ('Groq', groq_api_key), ('OpenRouter', openrouter_api_key)]:
    print(f"{name}: {'set' if key else 'not set'}")

In [ ]:
openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1/")
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
grok = OpenAI(api_key=grok_api_key, base_url="https://api.x.ai/v1")
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
openrouter = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

In [ ]:
models = [
    "gpt-5",
    "claude-sonnet-4-5-20250929",
    "grok-4",
    "gemini-2.5-pro",
    "qwen2.5-coder",
    "deepseek-coder-v2",
    "gpt-oss:20b",
    "qwen/qwen3-coder-30b-a3b-instruct",
    "openai/gpt-oss-120b",
]

clients = {
    "gpt-5": openai,
    "claude-sonnet-4-5-20250929": anthropic,
    "grok-4": grok,
    "gemini-2.5-pro": gemini,
    "openai/gpt-oss-120b": groq,
    "qwen2.5-coder": ollama,
    "deepseek-coder-v2": ollama,
    "gpt-oss:20b": ollama,
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter,
}

In [ ]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

In [ ]:
# Use the compile / run commands suggested by GPT-5 in day 3 — adjust for your platform.
compile_command = [
    "clang++", "-std=c++17", "-Ofast", "-mcpu=native",
    "-flto=thin", "-fvisibility=hidden", "-DNDEBUG",
    "main.cpp", "-o", "main"
]
run_command = ["./main"]

## Port logic

In [ ]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [ ]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [ ]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(
        model=model, messages=messages_for(python), reasoning_effort=reasoning_effort
    )
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp', '').replace('```', '')
    write_output(reply)
    return reply

## Test workload

In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    return output

In [ ]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        for _ in range(3):
            print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

## Gradio UI — pick a model from the dropdown and convert

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

In [ ]:
compile_and_run()